In [0]:
USE CATALOG training_medallion;
USE SCHEMA 00_landing;

In [0]:
CREATE OR REPLACE TABLE training_medallion.`01_bronze`.bronze_employee_incidents (
  incident_id STRING,
  employee_id STRING,
  incident_text STRING
) USING DELTA;

-- Masukkan beberapa baris data sampel
INSERT INTO training_medallion.`01_bronze`.bronze_employee_incidents VALUES
  ('INC-901', 'EMP-04', 'Karyawan mengakses database gaji eksekutif sebanyak 12 kali pada pukul 03.00 pagi tanpa izin.'),
  ('INC-902', 'EMP-12', 'Karyawan menyelesaikan pelatihan kepatuhan keamanan informasi tepat waktu.'),
  ('INC-903', 'EMP-88', 'Terdeteksi pengunduhan massal kode sumber ke flashdisk pribadi sebelum jam kantor berakhir.');

num_affected_rows,num_inserted_rows
3,3


In [0]:
CREATE OR REPLACE TABLE training_medallion.`02_silver`.silver_employee_incidents AS
WITH raw_ai AS (
  SELECT 
    incident_id,
    employee_id,
    incident_text,
    ai_query(
      'system.ai.meta-llama-3-1-8b-instruct',
      CONCAT(
        'Return ONLY a raw JSON object with keys "risk_tier" (Low, Medium, or Critical) and "action". Do not use markdown backticks or commentary. Text: ', 
        incident_text
      )
    ) AS ai_raw
  FROM training_medallion.`01_bronze`.bronze_employee_incidents
),
cleaned_ai AS (
  SELECT 
    incident_id,
    employee_id,
    incident_text,
    -- (?s) mengizinkan regex mencocokkan karakter newline multi-baris
    regexp_extract(ai_raw, '(?s)(\\{.*\\})', 1) AS clean_json
  FROM raw_ai
)
SELECT 
  incident_id,
  employee_id,
  incident_text,
  get_json_object(clean_json, '$.risk_tier') AS risk_tier,
  get_json_object(clean_json, '$.action') AS action
FROM cleaned_ai;

num_affected_rows,num_inserted_rows


In [0]:
select * from training_medallion.`02_silver`.silver_employee_incidents

incident_id,employee_id,incident_text,risk_tier,action
INC-901,EMP-04,Karyawan mengakses database gaji eksekutif sebanyak 12 kali pada pukul 03.00 pagi tanpa izin.,Critical,Investigasi dan tindakan tegas terhadap karyawan yang melanggar keamanan sistem
INC-902,EMP-12,Karyawan menyelesaikan pelatihan kepatuhan keamanan informasi tepat waktu.,Low,No action required
INC-903,EMP-88,Terdeteksi pengunduhan massal kode sumber ke flashdisk pribadi sebelum jam kantor berakhir.,Critical,Investigasi dan tindakan lanjutan


In [0]:
-- Pipeline Silver ➔ Gold (Agregasi Pelaporan Bisnis)
CREATE OR REPLACE TABLE training_medallion.`03_gold`.gold_risk_metrics_daily AS
SELECT 
  risk_tier,
  COUNT(incident_id) AS total_incidents,
  COUNT(DISTINCT employee_id) AS impacted_employees
FROM training_medallion.`02_silver`.silver_employee_incidents
GROUP BY risk_tier;

num_affected_rows,num_inserted_rows


In [0]:
select * from training_medallion.`03_gold`.gold_risk_metrics_daily

risk_tier,total_incidents,impacted_employees
Low,1,1
Critical,2,2
